# Summary Statistics & Significance Testing for ASSF Results

This notebook generates the per-sensor summary table (median R-CV, Unified Stability Score, stabilization index, adaptive MSD, standard deviations, 95% confidence intervals) and runs the Wilcoxon signed-rank test comparing conventional CV vs. Robust CV, in response to Reviewer 1's comment 4.8 ("Results remain predominantly qualitative").

**Instructions:** Update `CSV_PATH` below to point to your sensor data CSV, then run all cells.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

CSV_PATH = "Sensor_data_premonsoon.csv"   # <-- update this path to your dataset
SITE_COL = "Sample_ID"
SENSOR_COLS = ["dissolvedo2", "ph", "tds", "temp", "orp", "bga", "chlorophyll"]  # lowercase column names after normalization

## 1. Load data

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower() for c in df.columns]
site_col = SITE_COL.lower()

print("Shape:", df.shape)
print("Sites:", df[site_col].nunique())
df.head()

## 2. ASSF core functions

These reproduce the Adaptive Sensor Stability Framework pipeline: Robust Coefficient of Variation (R-CV), adaptive window selection, Adaptive Sensor Stabilization Detection (ASSD), adaptive Moving Standard Deviation (A-MSD), and the Unified Stability Score.

In [ ]:
def robust_cv(series, use_abs=False):
    """Median/MAD-based robust coefficient of variation (%)."""
    x = series.dropna().values.astype(float)
    if len(x) == 0:
        return np.nan
    median = np.median(x)
    if median == 0:
        return np.nan
    if use_abs:
        median = abs(median)
    mad = np.median(np.abs(x - np.median(x)))
    return float(mad / median * 100.0)


def conventional_cv(series):
    """Standard mean/std-based coefficient of variation (%)."""
    x = series.dropna().values.astype(float)
    if len(x) == 0:
        return np.nan
    mean, std = np.mean(x), np.std(x, ddof=1)
    if mean == 0:
        return np.nan
    return float(std / abs(mean) * 100.0)


def choose_adaptive_window(values, candidate_windows=(3, 5, 7, 11)):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < min(candidate_windows):
        return min(candidate_windows)
    best_w, best_score = candidate_windows[0], np.inf
    for w in candidate_windows:
        if len(x) < w:
            continue
        vars_ = [np.var(x[i:i + w]) for i in range(len(x) - w + 1)]
        mean_var = np.mean(vars_)
        global_var = np.var(x) if np.var(x) > 0 else 1.0
        score = mean_var / global_var
        if score < best_score:
            best_score, best_w = score, w
    return best_w


def learn_assd_threshold(series, window, base_quantile=0.25, min_threshold=0.005, max_threshold=0.05):
    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < window * 2:
        return 0.02
    z = (x - np.mean(x)) / (np.std(x) if np.std(x) > 0 else 1.0)
    vars_ = [np.var(z[i:i + window]) for i in range(len(z) - window + 1)]
    q_val = np.quantile(vars_, base_quantile)
    return float(np.clip(q_val, min_threshold, max_threshold))


def assd_adaptive(values, candidate_windows=(3, 5, 7, 11), consecutive=3):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return None, None, None
    window = choose_adaptive_window(x, candidate_windows)
    if len(x) < window + consecutive:
        return None, window, None
    thr = learn_assd_threshold(x, window)
    z = (x - np.mean(x)) / (np.std(x) if np.std(x) > 0 else 1.0)
    stable_count = 0
    for i in range(window, len(z)):
        v = np.var(z[i - window:i])
        if v < thr:
            stable_count += 1
            if stable_count >= consecutive:
                return i, window, thr
        else:
            stable_count = 0
    return None, window, thr


def moving_sd_adaptive(values, window):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < window:
        return np.nan
    sds = [np.std(x[i:i + window], ddof=1) for i in range(len(x) - window + 1)]
    return float(np.mean(sds)) if sds else np.nan


def stability_score(rcv, stable_idx, series_length):
    """Unified Stability Score: 0.5 * (1 - RCV term) + 0.5 * (1 - stabilization-speed term)."""
    if np.isnan(rcv) or series_length is None or series_length <= 1:
        return 0.0
    rcv_term = 1.0 - min(rcv, 30.0) / 30.0
    if stable_idx is None:
        stab_term = 0.0
    else:
        stab_term = 1.0 - min(stable_idx, series_length - 1) / float(series_length - 1)
    return float(max(0.0, min(1.0, 0.5 * rcv_term + 0.5 * stab_term)))


def classify(stable_idx, rcv):
    if stable_idx is None:
        return "High variability"
    if np.isnan(rcv):
        return "Stable (zero variation)"
    if rcv < 10:
        return "Operational-Stable"
    elif rcv < 30:
        return "Stable with natural variability"
    else:
        return "High variability"

## 3. Run the pipeline for every site x sensor combination

In [ ]:
rows = []
for site, site_df in df.groupby(site_col):
    for sensor in SENSOR_COLS:
        if sensor not in site_df:
            continue
        series = site_df[sensor].astype(float)
        use_abs = (sensor == "orp")

        rcv = robust_cv(series, use_abs)
        conv_cv = conventional_cv(series)
        stable_idx, win_used, thr_used = assd_adaptive(series.values)
        mean_msd = moving_sd_adaptive(series.values, win_used if win_used else 5)
        score = stability_score(rcv, stable_idx, len(series.dropna()))
        classification = classify(stable_idx, rcv)

        rows.append({
            "site": site,
            "sensor": sensor,
            "n_readings": len(series.dropna()),
            "mean": round(series.mean(), 3),
            "median": round(series.median(), 3),
            "std_dev": round(series.std(ddof=1), 3),
            "conventional_cv_%": round(conv_cv, 3) if conv_cv == conv_cv else np.nan,
            "robust_cv_%": round(rcv, 3) if rcv == rcv else np.nan,
            "assd_stable_index": stable_idx,
            "adaptive_msd": round(mean_msd, 3) if mean_msd == mean_msd else np.nan,
            "stability_score_0_1": round(score, 3),
            "classification": classification,
        })

detailed = pd.DataFrame(rows)
detailed.to_csv("assf_detailed_site_sensor_results.csv", index=False)
detailed.head(15)

## 4. Per-sensor summary table (Table V)

Median R-CV, Unified Stability Score (mean, SD, 95% CI), mean stabilization index, mean adaptive MSD, and their standard deviations, aggregated across all sites for each sensor. This is the table referenced in response to Reviewer 1 comment 4.8.

In [ ]:
summary_rows = []
for sensor, g in detailed.groupby("sensor"):
    rcv = g["robust_cv_%"].dropna()
    score = g["stability_score_0_1"].dropna()
    stab_idx = g["assd_stable_index"].dropna()
    msd = g["adaptive_msd"].dropna()

    n = len(score)
    mean_score = score.mean()
    sem = score.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    ci = stats.t.interval(0.95, n - 1, loc=mean_score, scale=sem) if n > 1 else (np.nan, np.nan)

    summary_rows.append({
        "sensor": sensor,
        "n_sites": n,
        "median_RCV_%": round(rcv.median(), 2),
        "RCV_std_%": round(rcv.std(ddof=1), 2),
        "mean_stability_score": round(mean_score, 3),
        "stability_score_std": round(score.std(ddof=1), 3),
        "stability_score_95CI_low": round(ci[0], 3),
        "stability_score_95CI_high": round(ci[1], 3),
        "mean_stabilization_index": round(stab_idx.mean(), 1),
        "stabilization_index_std": round(stab_idx.std(ddof=1), 1),
        "mean_adaptive_MSD": round(msd.mean(), 2),
        "adaptive_MSD_std": round(msd.std(ddof=1), 2),
    })

sensor_summary = pd.DataFrame(summary_rows)
sensor_summary.to_csv("sensor_summary_table.csv", index=False)
sensor_summary

## 5. Statistical hypothesis test: Robust CV vs. Conventional CV

A Wilcoxon signed-rank test (paired, non-parametric — appropriate given the skewed distributions) comparing conventional CV vs. Robust CV across all site x sensor combinations. This addresses the reviewer's comment that no statistical hypothesis testing was presented.

In [ ]:
paired = detailed.dropna(subset=["conventional_cv_%", "robust_cv_%"])
conv_cvs = paired["conventional_cv_%"].values
rob_cvs = paired["robust_cv_%"].values

stat, p_value = stats.wilcoxon(conv_cvs, rob_cvs)

print(f"n pairs (site x sensor): {len(conv_cvs)}")
print(f"Wilcoxon signed-rank test: statistic = {stat:.2f}, p-value = {p_value:.3e}")
print(f"Median conventional CV: {np.median(conv_cvs):.2f}%")
print(f"Median robust CV:       {np.median(rob_cvs):.2f}%")
print(f"Ratio (conventional / robust) per sensor:")
ratio_check = paired.groupby("sensor").apply(
    lambda g: (g["conventional_cv_%"].mean() / g["robust_cv_%"].mean())
)
print(ratio_check.round(2))

## 6. Suggested manuscript text

Once you've run this on your own dataset and confirmed the numbers, here is suggested wording for the Results and Discussion section:

> *"A Wilcoxon signed-rank test confirmed that Robust CV values were significantly lower than conventional CV values across all site-sensor combinations (n = &lt;N&gt;, W = &lt;STAT&gt;, p &lt; 0.001), with a median conventional CV of &lt;X&gt;% versus a median robust CV of &lt;Y&gt;%, supporting the use of the robust formulation for reducing sensitivity to outlier readings in field-deployed sensor data."*

Insert **Table V** (from `sensor_summary_table.csv`) into the Results and Discussion section, before Fig9 (Stability Score Heatmap), with a caption such as:

> *"Table V. Per-sensor summary statistics computed across all sites. Stability Score 95% CI computed via t-distribution. R-CV = Robust Coefficient of Variation."*